# Home Energy Helper RAG

A minimal retrieval-augmented generation (RAG) baseline for the Home Energy Helper. It searches Energy Saver guidance and asks the model to answer with source links. The Streamlit app will calculate monthly kWh and bill changes separately; RAG guidance must not guess what caused a change.

In [2]:
import json

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI()

## 1. Load Your Data

Possible knowledge sources for this project:

- Curated U.S. Department of Energy Energy Saver pages and guides (selected for this prototype): useful, practical guidance with stable source links, and quick to package as local documents.
- The U.S. Energy Information Administration (EIA) Open Data API: useful for regional electricity prices and trends; requires an API key and would add setup before the first demo.
- Synthetic monthly household bills: useful for testing charts and bill comparisons right away, but they cannot explain why a household's usage changed.

This prototype uses a local JSON file of short, topic-focused Energy Saver excerpts with source URLs. That keeps notebook runs repeatable and lets every guidance answer cite its source. The bill-comparison app should calculate kWh and cost changes separately; retrieved guidance cannot establish the cause of a usage change.

In [3]:
from pathlib import Path

data_path = Path("data/energy_saving_guides.json")
if not data_path.exists():
    data_path = Path("../data/energy_saving_guides.json")

with data_path.open(encoding="utf-8") as file:
    documents = json.load(file)

print(f"Loaded {len(documents)} Energy Saver documents from {data_path}")
print("Fields:", ", ".join(documents[0].keys()))

Loaded 5 Energy Saver documents from ..\data\energy_saving_guides.json
Fields: title, topic, content, source_name, source_url


## 2. Build the Index

minsearch builds an in-memory TF-IDF index over the fields you specify.

In [4]:
index = Index(
    text_fields=["title", "topic", "content"],
    keyword_fields=["source_name", "source_url"],
)

index.fit(documents)

## 3. Define the RAG Functions

`search` retrieves Energy Saver documents, `build_prompt` adds the question and source-linked context, `llm` calls the model, and `rag` chains those steps.

In [5]:
def search(query):
    return index.search(query, num_results=5)

In [6]:
instructions = """
You are a careful home energy helper. Use the retrieved U.S. Department of Energy context to answer questions about practical ways to reduce household electricity use. Give clear, actionable checks and distinguish general guidance from facts about this particular home.

Do not infer why a monthly kWh total changed: bills show the change, but not its cause. Do not calculate bill comparisons; the app handles those calculations. If the cause or a tailored recommendation depends on missing details, say what cannot be determined and ask a short, relevant follow-up question (for example, whether the billing period was longer or whether heating, a space heater, or air conditioning was used more).

Use only claims supported by the supplied context. Include a Markdown link to the source URL for every guidance-based answer, using the source title as the link text. If the context does not support an answer, say so and ask what information would help; do not invent a recommendation or source.
""".strip()


def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

    return user_prompt

In [7]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [8]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 4. Try It Out

Ask a practical question about what to check when electricity use changes. The answer should distinguish checks from a confirmed cause and link any energy-saving guidance to its DOE source.

In [9]:
print(rag("My electricity use increased in February. What should I check before assuming the furnace or a space heater caused it?"))

Before assuming that your furnace or a space heater caused the increase in electricity usage, you should check the following:

1. **Thermostat Settings and Schedules**: Review your thermostat settings. Investigate if they were set higher than usual or if the schedule changed, which could lead to increased heating usage.

2. **Use of Portable Heaters or ACs**: Consider whether you used a portable space heater or air conditioning unit more frequently during that period.

3. **Heating Equipment Operation**: Check if your heating equipment was running longer than usual, which can contribute to the increase in consumption.

4. **Filter Maintenance**: Ensure that the filters in your heating or cooling systems are clean or have been replaced, as dirty filters can reduce efficiency.

5. **Duct Inspections**: Look for any issues with duct leaks, as they can contribute to higher energy use. It's often recommended to have these inspected by a qualified professional.

6. **Air Leaks**: Evaluate yo